# Finite cluster x transverse to product stacked cluster interlayer coupling 200 site

Modification of [this notebook](finite_cluster_x_transverse_to_product_stacked_cluster_200_site.ipynb).

Take a stack of two chains. "Top" is of the form $$H = -(1-t)\sum_i Z_{i-1}X_i Z_{i+1} - t \sum_i X_i$$ and "bottom" of the form $$H = -\sum_i Z_{i-1}X_i Z_{i+1}$$. Also add coupling $H = -4t(1-t)\sum X_{i,1} X_{i,2}$, i.e. interlayer coupling.

The system is a $Z_2 \times Z_2^T$ SPT with symmetry generated by $X$ on the top layer and $XK$ on the bottom layer, where $K$ acts on both layers.

# Imports

In [1]:
import numpy as np
import scipy
import matplotlib.pyplot as plt

In [2]:
import tenpy
import tenpy.linalg.np_conserved as npc
from tenpy.algorithms import dmrg
from tenpy.networks.mps import MPS

In [3]:
from tenpy.networks.site import SpinSite, SpinHalfSite
from tenpy.models.lattice import Chain
from tenpy.models.model import CouplingModel, CouplingMPOModel, MPOModel

In [18]:
class StackedCluster(CouplingMPOModel):
    default_lattice = "Chain"
    force_default_lattice = True

    def init_sites(self, model_params):
        spin2_x = SpinHalfSite(conserve=None)
        spin2_xk = SpinHalfSite(conserve=None)
        return [spin2_x, spin2_xk], ['X', 'XK']

    def init_terms(self, model_params):
        t = model_params.get('t', 0)

        self.add_onsite(-1*t, 0, 'Sigmax')

        self.add_multi_coupling(
            -(1-t),
            [
                ('Sigmaz', -1, 0),
                ('Sigmax', 0, 0),
                ('Sigmaz', 1, 0),
            ]
        )

        self.add_multi_coupling(
            -1,
            [
                ('Sigmaz', -1, 1),
                ('Sigmax', 0, 1),
                ('Sigmaz', 1, 1),
            ]
        )

        self.add_multi_coupling(
            -4*(1-t)*t,
            [
                ('Sigmax', 0, 0),
                ('Sigmax', 0, 1),
            ]
        )

In [6]:
dmrg_params = {
    "trunc_params": {"chi_max": 16, "chi_min": 1, "svd_min": 1.e-8},
    "min_sweeps":10,
    "max_sweeps":200,
    "mixer": True,
    "combine":False,
    'decay':2,
    'amplitude':10e-1,
    'disable_after':60,
    'update_env':0
}

In [7]:
t_params = np.linspace(0, 1, 11)

In [8]:
t_params

array([0. , 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ])

In [9]:
import h5py
from tenpy.tools import hdf5_io

In [10]:
num_unit_cells=100

In [11]:
model=StackedCluster({'t': 0, 'L': num_unit_cells})

In [12]:
len(model.all_coupling_terms().to_TermList().terms)

196

In [13]:
model.all_onsite_terms().to_TermList().terms

[]

In [14]:
model=StackedCluster({'t': 1, 'L': num_unit_cells})

In [15]:
len(model.all_coupling_terms().to_TermList().terms)

98

In [16]:
len(model.all_onsite_terms().to_TermList().terms)

100

In [19]:
wavefunctions = list()
energies = list()

for t in t_params:
    print(f"Commencing interpolation={t} model")
    model=StackedCluster({'t': t, 'L': num_unit_cells})
    
    psi = MPS.from_lat_product_state(model.lat, [[0, 0],]*num_unit_cells)
    psi.canonical_form()

    eng = dmrg.TwoSiteDMRGEngine(psi, model, dmrg_params)
    e, psi = eng.run()

    print(f"Energy: {e}\n")

    data = {
        "wavefunction": psi,
        "energy": e,
        "paramters": {"interpolation": t}
    }

    t_string = str(int(100*t))
    one_minus_t_string = str(int(100*(1-t)))

    file_name = f'{t_string}'

    filename = (
        r"../data/finite_cluster_x_transverse_to_product_stacked_cluster_interlayer_coupling_200_site/{}"
        .format(file_name)
    )

    filename += ".h5"

    with h5py.File(filename, 'w') as f:

        hdf5_io.save_to_hdf5(f, data)

Commencing interpolation=0.0 model


/home/kcooney/Desktop/repos/spt_numerical_classificiation/code/num_spt_venv_p11/lib/python3.11/site-packages/tenpy/tools/params.py:230: UserWarning: unused options for config TwoSiteDMRGEngine:
['amplitude', 'decay', 'disable_after', 'update_env']
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))


Energy: -196.00000000000318

Commencing interpolation=0.1 model


/home/kcooney/Desktop/repos/spt_numerical_classificiation/code/num_spt_venv_p11/lib/python3.11/site-packages/tenpy/tools/params.py:230: UserWarning: unused options for config TwoSiteDMRGEngine:
['amplitude', 'decay', 'disable_after', 'update_env']
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=3.02e-05


Energy: -188.27338108969747

Commencing interpolation=0.2 model


...

Energy: -190.04425027548567

Commencing interpolation=0.9 model


/home/kcooney/Desktop/repos/spt_numerical_classificiation/code/num_spt_venv_p11/lib/python3.11/site-packages/tenpy/tools/params.py:230: UserWarning: unused options for config TwoSiteDMRGEngine:
['amplitude', 'decay', 'disable_after', 'update_env']
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=4.84e-05


Energy: -191.6616035465778

Commencing interpolation=1.0 model


/home/kcooney/Desktop/repos/spt_numerical_classificiation/code/num_spt_venv_p11/lib/python3.11/site-packages/tenpy/tools/params.py:230: UserWarning: unused options for config TwoSiteDMRGEngine:
['amplitude', 'decay', 'disable_after', 'update_env']
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))


Energy: -197.99999999999872

